# Advisor tool walkthrough (beta)

The advisor tool pairs a faster **executor** model with a
higher-intelligence **advisor** model. The executor decides when to
consult the advisor; Anthropic runs the advisor server-side over the
executor's full transcript and returns the advice as an
`advisor_tool_result` block, all inside a single `/v1/messages` request.

This notebook walks through: the quick start, inspecting the response
blocks, capping advisor output (the hard control), the brevity hint
(the soft control), reading the per-iteration usage breakdown,
multi-turn round-tripping, and the client-side conversation cap.

> **Beta note**: include the `advisor-tool-2026-03-01` beta header and
> confirm your key has access. The tool is on the Claude API and Claude
> Platform on AWS, not yet on Bedrock, Vertex AI, or Microsoft Foundry.

In [ ]:
import os
import sys

# Make the shared _advisor module importable whether the working directory is the repo
# root or this topic folder.
for _p in (".", "advisor"):
    if os.path.isfile(os.path.join(_p, "_advisor.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _advisor import (
    DEFAULT_ADVISOR,
    DEFAULT_EXECUTOR,
    DEFAULT_TASK,
    SUGGESTED_SYSTEM_PROMPT,
    count_advisor_calls,
    extract_advisor_events,
    final_text,
    get_client,
    make_advisor_tool,
    run_turn,
    strip_advisor_blocks,
    summarize_usage,
    with_brevity_hint,
)

client = get_client()
print(f"Executor: {DEFAULT_EXECUTOR}   Advisor: {DEFAULT_ADVISOR}")

## 1. Quick start

The tool definition carries the advisor `model`; the top-level `model`
is the executor. The executor and advisor must form a valid pair (the
advisor must be at least as capable as the executor), otherwise the API
returns a 400 naming the unsupported combination.

In [ ]:
tool = make_advisor_tool(model=DEFAULT_ADVISOR, max_uses=3, max_tokens=2048)
messages = [{"role": "user", "content": DEFAULT_TASK}]

response = run_turn(
    client,
    messages,
    executor=DEFAULT_EXECUTOR,
    tools=[tool],
    system=SUGGESTED_SYSTEM_PROMPT,
)
print(final_text(response)[:1500])

## 2. Inspect the blocks

A consultation appears as a `server_tool_use` block with
`name: "advisor"` and an **empty input** (the server builds the
advisor's view from the transcript; nothing the executor puts in input
reaches the advisor), followed by an `advisor_tool_result` block. The
result content is a discriminated union: `advisor_result` (plaintext),
`advisor_redacted_result` (encrypted; round-trip verbatim), or
`advisor_tool_result_error` (executor continues without advice).

In [ ]:
for block in response.content:
    print(getattr(block, "type", "?"), getattr(block, "name", ""))

In [ ]:
for event in extract_advisor_events(response):
    print(f"kind={event.kind}  stop_reason={event.stop_reason}")
    if event.text:
        print(event.text[:600], "\n")

## 3. The hard control: max_tokens on the tool definition

The top-level `max_tokens` bounds executor output only. The advisor's
own ceiling is `max_tokens` on the tool definition (minimum 1024).
Anthropic's testing found 2048 cut mean advisor output roughly 7x with
near-zero truncation. When a call does hit the cap, the result carries
`stop_reason: "max_tokens"`, so truncation is detectable rather than
silent. This is a govern-side control: a runtime ceiling the model
cannot talk its way past.

In [ ]:
capped = [e for e in extract_advisor_events(response) if e.stop_reason == "max_tokens"]
print(f"Advisor calls truncated at the cap: {len(capped)}")

## 4. The soft control: the brevity hint

The advisor sees the user message as quoted context, and direct address
works far better than third-person description. Prefixing a single
parenthetical line asking for under 80 words biases the advisor toward
short, focused guidance. It is a request, not a ceiling, so ask for
roughly 80 percent of your true limit. Prompts guide; the max_tokens
cap above governs. Use both together for the best cost-quality tradeoff.

In [ ]:
print(with_brevity_hint("Refactor the dedupe step to be streaming-friendly.")[:200])

## 5. Usage and billing: the iterations array

Advisor tokens are billed at the advisor model's rates, so they are NOT
rolled into the top-level usage totals. The full picture lives in
`usage.iterations[]`: entries with `type: "message"` are executor
iterations, entries with `type: "advisor_message"` are advisor
sub-inferences. If you build cost dashboards for agent workloads, this
array is the telemetry to capture.

In [ ]:
usage = summarize_usage(response, DEFAULT_EXECUTOR)
for i, row in enumerate(usage.rows, 1):
    print(
        f"{i}. {row.kind:<16} {row.model:<22} in={row.input_tokens:>6} "
        f"cache_read={row.cache_read:>6} out={row.output_tokens:>6}"
    )
print(
    f"\nexecutor output={usage.executor_output}  "
    f"advisor output={usage.advisor_output}  advisor calls={usage.advisor_calls}"
)

## 6. Multi-turn: round-trip the blocks

Append the full assistant content, including `advisor_tool_result`
blocks, back into history on later turns. Two failure modes to know:
omitting the advisor tool from `tools` while history still contains
advisor blocks returns a 400, and switching advisor models
mid-conversation can change the result variant, so branch on
`content.type`.

In [ ]:
messages.append({"role": "assistant", "content": response.content})
messages.append(
    {
        "role": "user",
        "content": "Add a CLI entry point and a small pytest for the dedupe window.",
    }
)
follow_up = run_turn(
    client,
    messages,
    executor=DEFAULT_EXECUTOR,
    tools=[tool],
    system=SUGGESTED_SYSTEM_PROMPT,
)
print(final_text(follow_up)[:800])

## 7. The conversation-level budget is yours to enforce

`max_uses` caps advisor calls per request, but there is no built-in
conversation-level cap. The pattern: count advisor calls client-side,
and once you hit your ceiling, remove the advisor tool from `tools`
AND strip the advisor blocks from history (otherwise: 400). This is
the platform-layer budget enforcement, the same control-plane move as
any other agent cost guardrail.

In [ ]:
messages.append({"role": "assistant", "content": follow_up.content})
calls_so_far = count_advisor_calls(messages)
print(f"Advisor calls so far in this conversation: {calls_so_far}")

CONVERSATION_CAP = 4
if calls_so_far >= CONVERSATION_CAP:
    messages = strip_advisor_blocks(messages)
    tools_for_next_turn = []
    print("Cap reached: advisor removed and history cleaned.")
else:
    tools_for_next_turn = [tool]
    print("Under the cap: advisor stays available.")

## 8. When to enable advisor-side caching

`caching: {"type": "ephemeral", "ttl": "5m"}` on the tool definition
caches the advisor's own transcript across calls in a conversation.
The advisor's prompt on call N is call N-1's prompt plus one segment,
so the prefix is stable. The write costs more than the reads save at
one or two calls; break-even is roughly three calls. Enable it for
long agent loops, leave it off for short tasks, and never toggle it
mid-conversation. One interaction to watch: `clear_thinking` with a
keep value other than "all" shifts the advisor's quoted transcript and
causes advisor-side cache misses (a cost issue, not a quality issue).

In [ ]:
cached_tool = make_advisor_tool(max_uses=8, max_tokens=2048, caching_ttl="5m")
print(cached_tool)

## Where this fits

The advisor tool is a control plane in miniature. The system prompt and
brevity line guide behavior; max_tokens, max_uses, and the client-side
conversation cap govern it; and usage.iterations makes the meter
visible. If you are preparing for the CCA exam, the decision points
here (executor-advisor pairing, hard versus soft controls, billing
attribution across two models in one request) are exactly the kind of
architecture tradeoffs worth being fluent in.